# 옵티마이저 3가지 비교
> 이번 글에서는 SGD, Adam, AdamW를 3가지 관점에서 비교해보려 합니다.
1. 기본적인 방식
2. 모멘텀
3. 가중치 감쇠

In [1]:
import torch

# 동일한 lr (\eta)과 w 감쇠율(\lambda) 사용을 위해 LR 및 WEIGHT_DECAY를 고정
LR = 0.1
WEIGHT_DECAY = 0.1

# 계산을 쉽게 보기 위한 momentum 효과 제거
# 여기에서 말하는 모멘텀은 Adam(W)의 \beta_1, \beta_2 를 말함 (SGD의 \mu는 기본적으로 안쓰기 때문에 제외)
BETAS = (0.0, 0.0)

# SGD의 가중치 감쇠 형식
SGD 방식은 가중치 감쇠를 할 때 단순하게 가중치에 기울기를 학습할 때 함께 빼주는 방식으로 구현됩니다.

$w_t = w_{t-1} - \mu \cdot (g_t + \lambda \cdot w_{t-1})$

이는 기존 가중치가 `1`, 기울기가 `0`, 가중치 감쇠율이 `0.1`, 학습률이 `0.1`일 경우에 결과가 `0.99`가 나옵ㄴ디ㅏ.

코드로 간단하게 확인하고 Adam과 AdamW의 방식을 비교하겠습니다.

In [12]:
# 가중치가 하나 있다고 가정
w_sgd = torch.nn.Parameter(torch.tensor([1.0]))

# 옵티마이저 생성
sgd = torch.optim.SGD(
    [w_sgd],
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

# grad를 0으로 초기화
w_sgd.grad = torch.tensor([0.0])

sgd.step()

print("w_sgd result:", w_sgd)

w_sgd result: Parameter containing:
tensor([0.9900], requires_grad=True)


# Adam과 weight_decay

Adam의 수식은 다음과 같이 가중치 감쇠율을 그대로 적용하는 방식이 아닙니다.


$m_t = \beta_1 \cdot m_{t-1} + (1-\beta_1) \cdot (g_t + \lambda \cdot w_t)$

<br>
$v_t = \beta_2 \cdot v_{t-1} + (1-\beta_2) \cdot (g_t + \lambda \cdot w_t)^2$

<br>
<br>

$w_t = w + \mu \cdot \frac{m_t}{\sqrt{v_t} + \epsilon}$


따라서 아래 결과를 보시면 weight decay가 `1 - 0.1`을 한 `0.9` 가 나오는 것이 아닌 `0.8999...`가 나오는 것을 확인할 수 있습니다.

In [5]:
# adam의 기본 가중치 (Paramter 타입)
w_adam = torch.nn.Parameter(torch.tensor([1.0]))

adam = torch.optim.Adam(
    [w_adam],                   # 옵티마이저가 추적하는 파라미터
    lr=LR,                      # Parameter에 적용시켜 줄 때 사용될 lr 값
    betas=BETAS,                # momentum과 velocity 계산에 사용될 \beta 값들
    weight_decay=WEIGHT_DECAY   # L2 규제와 동일한 형태로 적용되는 감쇠 방식
)

# 실제 loss에서 backward()를 하였을 때 발생한 gradient가 0이라고 가정
w_adam.grad = torch.tensor([0.0])

# L2 방식으로 \hat{m}와 \hat{v} gradient 통계값을 이용해서 weight 를 업데이트해주기
adam.step()

print("Adam weight_decay weight result:", w_adam.item())

Adam weight_decay weight result: 0.8999999761581421


# Adam + L2
Adam의 weight_decay 방식은 손실함수에 L2 규제를 가한 것과 동일한 결과를 낸다는 것을 나타내는 것을 증명합니다.

In [6]:
# Adam 옵티마이저를 이용하는 파라미터 생성
w_l2 = torch.nn.Parameter(torch.tensor([1.0]))

# w_l2 파라미터를 사용하는 optimizer 생성
# 위와 동일한 방식으로 설정하되 weight_decay는 0으로 설정하고 L2를 loss에 넣는 방식
adam_l2 = torch.optim.Adam(
    [w_l2],
    lr=LR,
    betas=BETAS,
    weight_decay=0.0
)

# L2 패널티를 가한 값은 L + (1/2)*w^2 이기 때문에 이를 미분한 값을 loss 로 설저
loss = WEIGHT_DECAY * (1 / 2) * (w_l2 ** 2).sum()

loss.backward()
adam_l2.step()

# 컴퓨터의 계산 방식 (부동소수점)에 의해 약간의 오차가 있지만 거의 동일
print("Adam L2 weight result:", w_l2.item())

Adam L2 weight result: 0.8999999761581421


# AdamW
AdamW는 정확하게 gradient를 계산한 후 거기에 $\lambda \times w$ 를 더하여 함께 빼주기 때문에 예상할 수 있는 형태인 $1 - 0.1 \times 0.1 = 0.99$ 가 결과로 나옵니다.

$m_t = \beta_1 \cdot m_{t-1} + (1-\beta_1) \cdot g_t$

<br>
$v_t = \beta_2 \cdot v_{t-1} + (1-\beta_2) \cdot g_t^2$

<br>
<br>

$w_t = w_{t-1} + \mu \cdot (\frac{m_t}{\sqrt{v_t} + \epsilon} + w_{t-1})$

In [7]:
w_adamW = torch.nn.Parameter(torch.tensor([1.0]))

adamW_optim = torch.optim.AdamW(
    [w_adamW],
    lr=LR,
    betas=BETAS,
    weight_decay=WEIGHT_DECAY
)

# 기울기를 계산에 넣지 않기 위해 초기화
w_adamW.grad = torch.tensor([0.0])

adamW_optim.step()

# 부동 소수점에 의해 정확히 0이 나오지는 않는 모습
print("AdamW weight result:", w_adamW.item())

AdamW weight result: 0.9900000095367432
